In [1]:
import numpy as np
import pandas as pd

import os

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.base import is_classifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

import joblib

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
# This is to be replaced with where the root folder of the project is located
ROOT_PATH = '/content/drive/Othercomputers/My MacBook Air/ml-research-dna-binding-protein/'
DATA_PATH = os.path.join(ROOT_PATH, 'datasets') # this is where I store the datasets
PSSM_PATH = os.path.join(DATA_PATH, 'pssm', 'pssm_outputs') # this is where I store pre-computed PSSMs
STACK_DP_PRED_PATH = os.path.join(ROOT_PATH, 'StackDPPred', 'src', 'paper_ver')
FEATURES_PATH = os.path.join(STACK_DP_PRED_PATH, 'train_code', 'features') # create if it does not exist
SAVED_MODELS_PATH = os.path.join(STACK_DP_PRED_PATH, 'saved_models') # create if it does not exist

os.makedirs(FEATURES_PATH, exist_ok=True)
os.makedirs(SAVED_MODELS_PATH, exist_ok=True)

In [4]:
# The usual default order, displayed as header of PSSM as well
AMINO_ACIDS = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
RCEM_RESIDUE_ORDER = sorted(AMINO_ACIDS)

In [5]:
# RCEM Matrix provided by author
ENERGY = np.array(
  [
    [-1.65,  -2.83, 1.16,	1.80,	-3.73,	-0.41,	1.90,	-3.69,	0.49,	-3.01,	-2.08,	0.66,	1.54,	1.20,	0.98, -0.08,  0.46, -2.31,	0.32,	-4.62],
    [-2.83,	-39.58,	-0.82,	-0.53,	-3.07,	-2.96,	-4.98,	0.34,	-1.38,	-2.15,	1.43,	-4.18,	-2.13,	-2.91,	-0.41,	-2.33,	-1.84,	-0.16,	4.26,	-4.46],
    [1.16,	-0.82,	0.84,	1.97,	-0.92,	0.88,	-1.07,	0.68,	-1.93,	0.23,	0.61,	0.32,	3.31,	2.67,	-2.02,	0.91,	-0.65,	0.94,	-0.71,	0.90],
    [1.80,	-0.53,	1.97,	1.45,	0.94,	1.31,	0.61,	1.30,	-2.51,	1.14,	2.53,	0.20,	1.44,	0.10,	-3.13,	0.81,	1.54,	0.12,	-1.07,	1.29],
    [-3.73,	-3.07,	-0.92,	0.94,	-11.25,	0.35,	-3.57,	-5.88,	-0.82,	-8.59,	-5.34,	0.73,	0.32,	0.77,	-0.40,	-2.22,	0.11,	-7.05,	-7.09,	-8.80],
    [-0.41,	-2.96,	0.88,	1.31,	0.35,	-0.20,	1.09,	-0.65,	-0.16,	-0.55,	-0.52,	-0.32,	2.25,	1.11,	0.84,	0.71,	0.59,	-0.38,	1.69,	-1.90],
    [1.90,	-4.98,	-1.07,	0.61,	-3.57,	1.09,	1.97,	-0.71,	2.89,	-0.86,	-0.75,	1.84,	0.35,	2.64,	2.05,	0.82,	-0.01,	0.27,	-7.58,	-3.20],
    [-3.69,	0.34,	0.68,	1.30,	-5.88,	-0.65,	-0.71,	-6.74,	-0.01,	-9.01,	-3.62,	-0.07,	0.12,	-0.18,	0.19,	-0.15,	0.63,	-6.54,	-3.78,	-5.26],
    [0.49,	-1.38,	-1.93,	-2.51,	-0.82,	-0.16,	2.89,	-0.01,	1.24,	0.49,	1.61,	1.12,	0.51,	0.43,	2.34,	0.19,	-1.11,	0.19,	0.02,	-1.19],
    [-3.01,	-2.15,	0.23,	1.14,	-8.59,	-0.55,	-0.86,	-9.01,	0.49,	-6.37,	-2.88,	0.97,	1.81,	-0.58,	-0.60,	-0.41,	0.72,	-5.43,	-8.31,	-4.90],
    [-2.08,	1.43,	0.61,	2.53,	-5.34,	-0.52,	-0.75,	-3.62,	1.61,	-2.88,	-6.49,	0.21,	0.75,	1.90,	2.09,	1.39,	0.63,	-2.59,	-6.88,	-9.73],
    [0.66,	-4.18,	0.32,	0.20,	0.73,	-0.32,	1.84,	-0.07,	1.12,	0.97,	0.21,	0.61,	1.15,	1.28,	1.08,	0.29,	0.46,	0.93,	-0.74,	0.93],
    [1.54,	-2.13,	3.31,	1.44,	0.32,	2.25,	0.35,	0.12,	0.51,	1.81,	0.75,	1.15,	-0.42,	2.97,	1.06,	1.12,	1.65,	0.38,	-2.06,	-2.09],
    [1.20,	-2.91,	2.67,	0.10,	0.77,	1.11,	2.64,	-0.18,	0.43,	-0.58,	1.90,	1.28,	2.97,	-1.54,	0.91,	0.85,	-0.07,	-1.91,	-0.76,	0.01],
    [0.98,	-0.41,	-2.02,	-3.13,	-0.40,	0.84,	2.05,	0.19,	2.34,	-0.60,	2.09,	1.08,	1.06,	0.91,	0.21,	0.95,	0.98,	0.08,	-5.89,	0.36],
    [-0.08,	-2.33,	0.91,	0.81,	-2.22,	0.71,	0.82,	-0.15,	0.19,	-0.41,	1.39,	0.29,	1.12,	0.85,	0.95,	-0.48,	-0.06,	0.13,	-3.03,	-0.82],
    [0.46,	-1.84,	-0.65,	1.54,	0.11,	0.59,	-0.01,	0.63,	-1.11,	0.72,	0.63,	0.46,	1.65,	-0.07,	0.98,	-0.06,	-0.96,	1.14,	-0.65,	-0.37],
    [-2.31,	-0.16,	0.94,	0.12,	-7.05,	-0.38,	0.27,	-6.54,	0.19,	-5.43,	-2.59,	0.93,	0.38,	-1.91,	0.08,	0.13,	1.14,	-4.82,	-2.13,	-3.59],
    [0.32,	4.26,	-0.71,	-1.07,	-7.09,	1.69,	-7.58,	-3.78,	0.02,	-8.31,	-6.88,	-0.74,	-2.06,	-0.76,	-5.89,	-3.03,	-0.65,	-2.13,	-1.73,	-12.39],
    [-4.62,	-4.46,	0.90,	1.29,	-8.80,	-1.90,	-3.20,	-5.26,	-1.19,	-4.90,	-9.73,	0.93,	-2.09,	0.01,	0.36,	-0.82,	-0.37,	-3.59,	-12.39,	-2.68],
  ])

# Feature Extraction

In [6]:
# Returns a list of tuples, with the first element as the amino acid and the 2nd element as a numpy array of the corresponding 20 probabilities
def parse_pssm_arr(file_path):
  with open(file_path, "r") as f:
    lines = f.readlines()

  data = []
  for i in range(3, len(lines) - 6):
    parsed = lines[i].split()
    acid, values = parsed[1], parsed[2:22]
    np_val = np.array(list(map(int, values)), dtype=np.float64)
    data.append((acid, np_val))
  return data

def split_residue_pssm(pssm_data):
  seq = [data[0] for data in pssm_data]
  pssm = [data[1] for data in pssm_data]
  return seq, np.array(pssm)

In [7]:
def pssm_sdt(pssm_matrix, K=5):
  # consider separated bigrams of distance 1, 2, ..., K
  pssm_sdt = [np.zeros(20, dtype=np.float64) for _ in range(K)]
  len_seq = len(pssm_matrix)
  for k in range(1, K+1): # k = 1 means adjacent pairs, k=2 corresponds to amino acid pairs that are separated by 1 amino acid in between.
    for i in range(len_seq - k):
      pssm_sdt[k-1] += pssm_matrix[i] * pssm_matrix[i+k]
    pssm_sdt[k-1] /= (len_seq - k)
  return np.concatenate(pssm_sdt)

In [ ]:
# This works too and its much neater imo.
# def pssm_sdt(pssm_matrix, K=5):
#   pssm_sdt = []
#   len_seq = len(pssm_matrix)
#   for k in range(1, K+1):
#     first_part = pssm_matrix[:-k]
#     second_part = pssm_matrix[k:]
#     pssm_sdt.append(np.sum(first_part*second_part, axis=0)/(len_seq-k))
#   return np.array(pssm_sdt).flatten()

# _, pssm = split_residue_pssm(parse_pssm_arr(os.path.join(PSSM_PATH, 'pdb_186_pssm/1.pssm')))
# pssm_sdt(pssm) == pssm_sdt_alt(pssm)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

In [8]:
def pssm_ddt(pssm_matrix, K=5):
  pssm_ddt = []
  len_seq = len(pssm_matrix)
  for k in range(1, K+1):
    first_part = pssm_matrix[ :-k]
    second_part = pssm_matrix[k: ]
    cum = []
    for i in range(1, 20): # shift second_part to left from once up till 19 times
      shifted_second = np.roll(second_part, shift=-i, axis=1)
      res = np.sum(first_part * shifted_second, axis=0) / (len_seq - k)
      cum.append(res)
    pssm_ddt.append(np.concatenate(cum))
  return np.concatenate(pssm_ddt)

In [9]:
# Best interpretation of what was described in paper; Sum up rows of values of represented by the same residue type
def residue_probing_transformation(sequence, pssm_matrix):
  l = len(pssm_matrix)
  amino_acids = set(AMINO_ACIDS)
  cumulative_sums = {acid : np.zeros(20, dtype=np.float64) for acid in amino_acids}

  for i, acid in enumerate(sequence):
    if acid not in amino_acids:
      print(f"Found invalid or ambiguous character {acid}. This character will be ignored during the computatation of RPT.")
      continue
    assert acid in amino_acids, "This should not occurr during the computation of RPT."
    cumulative_sums[acid] += pssm_matrix[i]

  rpt = []
  for acid in AMINO_ACIDS: # stick to this order
    rpt.append(cumulative_sums[acid] / l)
  return np.stack(rpt, axis=0).flatten()


In [ ]:
# test rpt
# seq, pssm = split_residue_pssm(parse_pssm_arr(os.path.join(PSSM_PATH, 'pdb_186_pssm/1.pssm')))
# print(residue_probing_transformation(seq, pssm).shape)
# residue_probing_transformation(seq, pssm)

In [10]:
def evolutionary_distance_transformation(pssm_matrix, D=30): # need to make sure seq_len > 30
  len_seq = len(pssm_matrix)
  assert len_seq > 30, "Unable to run EDT because sequence received has length <= 30"
  edt = []
  for d in range(1, D+1):
    first_part = pssm_matrix[:-d]
    second_part = pssm_matrix[d:]
    res = 0
    for i in range(20):
      shifted_second = np.roll(second_part, shift=-i, axis=1)
      res += np.sum(np.square(first_part - shifted_second), axis=0) / (len_seq - d)
    edt.append(res)
  return np.concatenate(edt)

In [ ]:
# THIS clearly shows the need for feature scaling

# _, pssm = split_residue_pssm(parse_pssm_arr(os.path.join(PSSM_PATH, 'pdb_186_pssm/1.pssm')))
# print(evolutionary_distance_transformation(pssm).shape)
# evolutionary_distance_transformation(pssm)

(600,)


array([159.60810811, 210.09459459, 191.60810811, 249.12162162,
       182.01351351, 181.60810811, 219.82432432, 204.90540541,
       187.82432432, 224.01351351, 234.95945946, 193.06756757,
       185.55405405, 210.58108108, 214.17567568, 159.93243243,
       157.01351351, 229.5       , 205.66216216, 185.82432432,
       169.42465753, 216.93150685, 205.09589041, 249.47945205,
       182.82191781, 191.78082192, 239.15068493, 210.71232877,
       193.42465753, 211.7260274 , 227.64383562, 211.26027397,
       179.45205479, 209.5890411 , 217.28767123, 171.97260274,
       162.76712329, 228.63013699, 203.42465753, 176.54794521,
       159.66666667, 218.13888889, 202.94444444, 243.69444444,
       184.72222222, 178.83333333, 229.61111111, 202.94444444,
       185.16666667, 215.69444444, 235.25      , 200.88888889,
       183.52777778, 210.16666667, 214.41666667, 161.05555556,
       158.19444444, 228.97222222, 208.13888889, 177.86111111,
       165.30985915, 207.90140845, 191.98591549, 226.35

In [11]:
def rcem_transformation(sequence, order=RCEM_RESIDUE_ORDER, table=ENERGY):
  rcemt = []
  known_20 = set(AMINO_ACIDS)
  for residue in sequence:
    if residue not in known_20:
      print(f'Found invalid or ambiguous residue: {residue}. RCEMT will ignore this character.')
      continue
    rcemt.append(table[order.index(residue)]) # append the row associated to this residue as determined by the table
  rcemt = np.stack(rcemt) # convert list of numpy arrays to numpy array of arrays
  rcemt = np.sum(rcemt, axis=0) # sum column-wise
  rcemt = rcemt / np.sum(rcemt)
  return rcemt

**Custom Transformer**

Combines all the feature extraction steps into 1 module, 2820 features should be obtained.

In [12]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    ret = []
    c = 0
    for x in X:
      seq, pssm = split_residue_pssm(x)
      if c % 100 == 0:
        print(f"Processed {c}.")
      pssm_sdt_ = pssm_sdt(pssm)
      pssm_ddt_ = pssm_ddt(pssm)
      rpt = residue_probing_transformation(seq, pssm)
      edt = evolutionary_distance_transformation(pssm)
      rcemt = rcem_transformation(seq)
      res = np.concatenate((pssm_sdt_, pssm_ddt_, rpt, edt, rcemt))
      c += 1
      ret.append(res)
    print(f"Completed extraction for batch of {c}.")
    return np.stack(ret)

**PDB dataset**

train - 1075 (525 positive, 550 negative)

test - 186 (93 positive, 93 negative)

**Additional dataset**

positive - 7128

negative - 7133

In [13]:
def get_all_file_paths_in_pssm_folder(relative_folder_path):
  folder_path = os.path.join(PSSM_PATH, relative_folder_path)
  file_names = sorted(os.listdir(folder_path),
                      key = lambda x : int(x.split('.')[0]))
  file_paths = []
  for file in file_names:
    file_paths.append(os.path.join(folder_path, file))
  return file_paths

In [14]:
def get_pdb1075_file_paths():
  return get_all_file_paths_in_pssm_folder('pdb_1075_pssm/')

def get_pdb186_file_paths():
  return get_all_file_paths_in_pssm_folder('pdb_186_pssm/')

def get_7kbinding_file_paths():
  return get_all_file_paths_in_pssm_folder('7k_binding_pssm/')

def get_7knonbinding_file_paths():
  return get_all_file_paths_in_pssm_folder('7k_non_binding_pssm/')

In [15]:
# Files whose numbers range from 1 - 525 have positive labels. But not all files are present so need to find the actual index.
def extract_pdb1075_pssm():
  pssm_data = []
  all_files = get_pdb1075_file_paths()
  num_positive = 0
  for i, file in enumerate(all_files):
    pssm_data.append(parse_pssm_arr(file))
    _, file_name = os.path.split(file)
    if int(file_name.split('.')[0]) <= 525: # get the number associated with the file.
      num_positive += 1
  labels = np.concatenate((np.ones(num_positive), np.zeros(len(all_files) - num_positive)))
  return pssm_data, labels


# Files whose numbers range from 1 - 93 have positive labels.
def extract_pdb186_pssm():
  pssm_data = []
  all_files = get_pdb186_file_paths()
  num_positive = 0
  for i, file in enumerate(all_files):
    pssm_data.append(parse_pssm_arr(file))
    _, file_name = os.path.split(file)
    if int(file_name.split('.')[0]) <= 93: # get the number associated with the file.
      num_positive += 1
  labels = np.concatenate((np.ones(num_positive), np.zeros(len(all_files) - num_positive)))
  return pssm_data, labels


def extract_7kbinding_pssm():
  pssm_data = []
  all_files = get_7kbinding_file_paths()
  for file in all_files:
    pssm_data.append(parse_pssm_arr(file))
  labels = np.ones(len(pssm_data))
  return pssm_data, labels


def extract_7knonbinding_pssm():
  pssm_data = []
  all_files = get_7knonbinding_file_paths()
  for file in all_files:
    pssm_data.append(parse_pssm_arr(file))
  labels = np.zeros(len(pssm_data))
  return pssm_data, labels


# Combines binding and non-binding pssms into one file
def extract_additional_pssm():
  pssm_data_pos, labels_pos = extract_7kbinding_pssm()
  pssm_data_neg, labels_neg = extract_7knonbinding_pssm()
  pssm_combined = pssm_data_pos + pssm_data_neg
  labels = np.concatenate((labels_pos, labels_neg))
  return pssm_combined, labels

# Get Data

Retrieve if features previously extracted and saved, else run.

In [16]:
pdb_1075_features_path = os.path.join(FEATURES_PATH, "pdb_1075_features.npy")
pdb_186_features_path = os.path.join(FEATURES_PATH, "pdb_186_features.npy")
additional_combined_features_path = os.path.join(FEATURES_PATH, "additional_combined_features.npy")

In [17]:
def shuffle(data, labels):
  num_samples = data.shape[0]
  shuffled_indices = np.random.permutation(num_samples)
  return data[shuffled_indices], labels[shuffled_indices]

def combine_features_labels(features, labels):
  return np.c_[features, labels]

def load_data(path):
  if os.path.exists(path):
    data = np.load(path)
    num_features = data.shape[1] - 1
    X = data[:, :num_features]
    y = data[:, num_features]
  else:
    features, labels = None, None

    if path == pdb_1075_features_path:
      sequences, labels = extract_pdb1075_pssm()

    elif path == pdb_186_features_path:
      sequences, labels = extract_pdb186_pssm()

    elif path == additional_combined_features_path:
      sequences, labels = extract_additional_pssm()

    else:
      print("Path not recognized.")
      return
    extractor = FeatureExtractor()
    features = extractor.fit_transform(sequences)
    features_labels = combine_features_labels(features, labels)
    np.save(path, features_labels)
    num_features = features_labels.shape[1] - 1
    X = features_labels[:, :num_features]
    y = features_labels[:, num_features]
  return shuffle(X, y)

In [18]:
X_pdb1075, y_pdb1075 = load_data(pdb_1075_features_path)
X_pdb186, y_pdb186 = load_data(pdb_186_features_path)

X_additional, y_additional = load_data(additional_combined_features_path)

Processed 0.
Processed 100.
Processed 200.
Processed 300.
Processed 400.
Processed 500.
Processed 600.
Found invalid or ambiguous character X. This character will be ignored during the computatation of RPT.
Found invalid or ambiguous residue: X. RCEMT will ignore this character.
Processed 700.
Found invalid or ambiguous character X. This character will be ignored during the computatation of RPT.
Found invalid or ambiguous residue: X. RCEMT will ignore this character.
Found invalid or ambiguous character X. This character will be ignored during the computatation of RPT.
Found invalid or ambiguous residue: X. RCEMT will ignore this character.
Processed 800.
Processed 900.
Found invalid or ambiguous character X. This character will be ignored during the computatation of RPT.
Found invalid or ambiguous residue: X. RCEMT will ignore this character.
Found invalid or ambiguous character X. This character will be ignored during the computatation of RPT.
Found invalid or ambiguous residue: X. R

**Save Feature Scaler**

StandardScaler is fitted on PDB1075 which is the dataset the saved model is trained on to do predictions as per paper.

In [21]:
scaler_path = os.path.join(SAVED_MODELS_PATH, "scaler.pkl")

def train_scaler(data_X, save_scaler=True, trained_scaler_path=scaler_path):
  scaler = StandardScaler()
  scaler.fit(data_X)
  if save_scaler:
    joblib.dump(scaler, trained_scaler_path)

train_scaler(X_pdb1075)

**Pick train and test sets**

This helper will perform fit the feature scaler on the train set and apply on the test set.



In [22]:
def feature_scale_train_test(train_X, test_X):
  scaler = StandardScaler()
  scaled_train_X = scaler.fit_transform(train_X)
  scaled_test_X = scaler.transform(test_X)
  return scaled_train_X, scaled_test_X

# Train Model

An ensemble method known as Stacking is applied.

Base classifiers:
1. SVM
2. LogisticRegression
3. K-Nearest Neighbors
4. Random DecisionForest

Meta classifier:
1. SVM

In [23]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning, module='sklearn.linear_model._logistic')

def default_base_clf():
  return [SVC(probability=True), LogisticRegression(), KNeighborsClassifier(), RandomForestClassifier()]
def default_meta_clf():
  return SVC(probability=True)

In [24]:
class PM:
  num_dp = 4

  @staticmethod
  def calculate_precision(tp, fp):
    return round(tp / (tp + fp), PM.num_dp)

  @staticmethod
  def calculate_recall(tp, fn):
    return round(tp / (tp + fn), PM.num_dp)

  @staticmethod
  def calculate_accuracy(nc, total):
    return round(nc / total, PM.num_dp)

  @staticmethod
  def calculate_MCC(tp, tn, fp, fn):
    return round( (tp*tn - fp*fn) / ((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))**0.5, PM.num_dp)

  @staticmethod
  def calculate_F1(tp, fp, fn):
    return round(tp / (tp + (fn + fp)/2), PM.num_dp)

  @staticmethod
  def calculate_specificity(tn, fp):
    return round(tn / (tn + fp), PM.num_dp)

  @staticmethod
  def calculate_sensitivity(tp, fn):
    return PM.calculate_recall(tp, fn)

In [ ]:
# svm = SVC(probability=True)
# logreg = LogisticRegression()
# knn = KNeighborsClassifier()
# rf = RandomForestClassifier()

# svm.fit(X_pdb1075, y_pdb1075)
# logreg.fit(X_pdb1075, y_pdb1075)
# knn.fit(X_pdb1075, y_pdb1075)
# rf.fit(X_pdb1075, y_pdb1075)

# print(svm.predict([X_pdb186[110]]))
# print(logreg.predict([X_pdb186[110]]))
# print(knn.predict([X_pdb186[110]]))
# rf.predict([X_pdb186[110]])

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[0.]
[1.]
[0.]


array([0.])

In [ ]:
# print(svm.predict_proba([X_pdb186[110]]))
# print(logreg.predict_proba([X_pdb186[110]]))
# print(knn.predict_proba([X_pdb186[110]]))
# rf.predict_proba([X_pdb186[0]])

[[0.75351477 0.24648523]]
[[0.15716181 0.84283819]]
[[1. 0.]]


array([[0.13, 0.87]])

In [25]:
class StackDPPred(BaseEstimator): # gives set_params and get_params method
  def __init__(self, base_clf = [], meta_clf = None):
    assert base_clf != [], "Model needs to be initialized with base classifiers."
    assert meta_clf != None, "Model needs to be intialized with 1 meta classifier."

    for clf in base_clf:
      assert is_classifier(clf), f"base classifier, {clf} declared is not a classifier."
      assert hasattr(clf, 'predict_proba'), f"base classifier, {clf} declared does not support predict_proba method"

    assert is_classifier(meta_clf), f"{meta_clf} declared is not a classifier."
    assert hasattr(meta_clf, 'predict_proba'), f"{meta_clf} declared does not support predict_proba method"

    self.base_clf = base_clf
    self.meta_clf = meta_clf

  def fit(self, X=None, y=None):
    all_probas = []
    X, y = shuffle(X, y) # shuffle again for good measure because why not
    for clf in self.base_clf: # train on base_clf
      clf.fit(X, y)
      clf_res_probas = clf.predict_proba(X)
      all_probas.append(clf_res_probas)
    # combine input features with binding and non-binding probas
    X_combined = np.concatenate(
        [X] + all_probas,
        axis=1
    )
    # train meta_clf
    self.meta_clf.fit(X_combined, y)

  def predict(self, X):
    all_probas = []
    for clf in self.base_clf:
      clf_res_probas = clf.predict_proba(X)
      all_probas.append(clf_res_probas)
    X_combined = np.concatenate(
        [X] + all_probas,
        axis=1
    )
    return self.meta_clf.predict(X_combined)

  def predict_proba(self, X):
    all_probas = []
    for clf in self.base_clf:
      clf_res_probas = clf.predict_proba(X)
      all_probas.append(clf_res_probas)
    X_combined = np.concatenate(
        [X] + all_probas,
        axis=1
    )
    return self.meta_clf.predict_proba(X_combined)

  def save_model(self, saved_models_folder=SAVED_MODELS_PATH):
    # check if it has been trained
    if hasattr(self.meta_clf, 'support_'):
      for base_clf in self.base_clf:
        clf_path = os.path.join(saved_models_folder, "base_" + base_clf.__class__.__name__)
        joblib.dump(base_clf, clf_path)
      meta_clf_path = os.path.join(saved_models_folder, "meta_" + self.meta_clf.__class__.__name__)
      joblib.dump(self.meta_clf, meta_clf_path)
    else:
      print("Model not yet trained. Please call method after fitting.")

  def load_model(self, saved_models_folder=SAVED_MODELS_PATH):
    for i, base_clf in enumerate(self.base_clf):
      file_name = "base_" + base_clf.__class__.__name__
      clf_path = os.path.join(saved_models_folder, file_name)
      assert os.path.exists(clf_path), f"{file_name} does not exist. Please train first."
      self.base_clf[i] = joblib.load(clf_path) # replace with trained

    meta_file_name = "meta_" + self.meta_clf.__class__.__name__
    meta_clf_path = os.path.join(saved_models_folder, meta_file_name)
    assert os.path.exists(meta_clf_path), f"{meta_file_name} does not exist. Please train first."
    self.meta_clf = joblib.load(meta_clf_path) # replace with trained

### Train on PDB 1075 and evaluate on PDB186

Save this model.

In [26]:
ensemble = StackDPPred(default_base_clf(), default_meta_clf())

X_pdb1075_scaled, X_pdb186_scaled = feature_scale_train_test(X_pdb1075, X_pdb186)
ensemble.fit(X_pdb1075_scaled, y_pdb1075)

In [28]:
# Display results
y_pred186 = ensemble.predict(X_pdb186_scaled)
tn, fp, fn, tp = confusion_matrix(y_pdb186, y_pred186).ravel()

print("Below are results on PDB186 after training on PDB1075:")
print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")

Below are results on PDB186 after training on PDB1075:
Accuracy: 0.7386
Sensitivity: 0.9195
Specificity: 0.5618
MCC: 0.5143


In [30]:
# save
ensemble.save_model()

In [ ]:
# Test if saving works

# saved_ensemble = StackDPPred(default_base_clf(), default_meta_clf())
# saved_ensemble.load_model()

# # Display results
# y_pred = saved_ensemble.predict(X_pdb186_scaled)
# tn, fp, fn, tp = confusion_matrix(y_pdb186, y_pred).ravel()

# print("Below are results on PDB186 after training on PDB1075:")
# print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
# print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
# print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
# print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")

Below are results on PDB186 after training on PDB1075:
Accuracy: 0.7443
Sensitivity: 0.931
Specificity: 0.5618
MCC: 0.529


### Train on PDB1075 and test on PDB14k

In [ ]:
# Note model trained above
_, X_additional_scaled = feature_scale_train_test(X_pdb1075, X_additional)

# Display results
y_pred14k = ensemble.predict(X_additional_scaled)
tn, fp, fn, tp = confusion_matrix(y_additional, y_pred14k).ravel()

print("Below are results on PDB14k after training on PDB1075:")
print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")

Below are results on PDB14k after training on PDB1075:
Accuracy: 0.8168
Sensitivity: 0.9008
Specificity: 0.7328
MCC: 0.6427


### Train on PDB14k and evaluate on PDB1075 and PDB186


In [ ]:
ensemble_14k = StackDPPred(default_base_clf(), default_meta_clf())

# On PDB1075
X_additional_scaled, X_pdb1075_scaled = feature_scale_train_test(X_additional, X_pdb1075)
ensemble_14k.fit(X_additional_scaled, y_additional)

# Display results
y_pred1075 = ensemble_14k.predict(X_pdb1075_scaled)
tn, fp, fn, tp = confusion_matrix(y_pdb1075, y_pred1075).ravel()

print("Below are results of ensemble_14k on PDB1075 after training on PDB14k:")
print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")


# On PDB186. Note model is trained above
_, X_pdb186_scaled = feature_scale_train_test(X_additional, X_pdb186)

# Display results
y_pred186 = ensemble_14k.predict(X_pdb186_scaled)
tn, fp, fn, tp = confusion_matrix(y_pdb186, y_pred186).ravel()

print("Below are results on PDB186 after training on PDB14k:")
print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")

Below are results of ensemble_14k on PDB1075 after training on PDB14k:
Accuracy: 0.7724
Sensitivity: 0.7918
Specificity: 0.7547
MCC: 0.546
Below are results on PDB186 after training on PDB14k:
Accuracy: 0.7557
Sensitivity: 0.8276
Specificity: 0.6854
MCC: 0.5178


### Try training with SVM only:
1. On PDB1075 and evaluate on PDB186 and PDB14k
2. On PDB14k and evaluate on PDB1075 and PDB186

In [ ]:
svm_only = SVC(probability=True)

X_pdb1075_scaled, X_pdb186_scaled = feature_scale_train_test(X_pdb1075, X_pdb186)
svm_only.fit(X_pdb1075_scaled, y_pdb1075)

# Display results on PDB186
y_pred186 = svm_only.predict(X_pdb186_scaled)
tn, fp, fn, tp = confusion_matrix(y_pdb186, y_pred186).ravel()

print("Below are results of svm_only clf on PDB186 after training on PDB1075:")
print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")


# Display results on PDB14K
_, X_additional_scaled = feature_scale_train_test(X_pdb1075, X_additional)

y_pred14k = svm_only.predict(X_additional_scaled)
tn, fp, fn, tp = confusion_matrix(y_additional, y_pred14k).ravel()

print("Below are results on PDB14k after training on PDB1075:")
print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")

Below are results of svm_only clf on PDB186 after training on PDB1075:
Accuracy: 0.733
Sensitivity: 0.908
Specificity: 0.5618
MCC: 0.4998
Below are results on PDB14k after training on PDB1075:
Accuracy: 0.8145
Sensitivity: 0.9017
Specificity: 0.7273
MCC: 0.6387


In [ ]:
svm_only_14k = SVC(probability=True)

X_additional_scaled, X_pdb1075_scaled = feature_scale_train_test(X_additional, X_pdb1075)
svm_only_14k.fit(X_additional_scaled, y_additional)

# Display results on PDB1075
y_pred1075 = svm_only_14k.predict(X_pdb1075_scaled)
tn, fp, fn, tp = confusion_matrix(y_pdb1075, y_pred1075).ravel()

print("Below are results of svm_only_14k clf on PDB1075 after training on PDB14k:")
print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")


# Display results on PDB186
_, X_pdb186_scaled = feature_scale_train_test(X_additional, X_pdb186)

y_pred186 = svm_only_14k.predict(X_pdb186_scaled)
tn, fp, fn, tp = confusion_matrix(y_pdb186, y_pred186).ravel()

print("Below are results on PDB186 after training on PDB14k:")
print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")

Below are results of svm_only_14k clf on PDB1075 after training on PDB14k:
Accuracy: 0.7448
Sensitivity: 0.8062
Specificity: 0.6887
MCC: 0.4967
Below are results on PDB186 after training on PDB14k:
Accuracy: 0.7386
Sensitivity: 0.8391
Specificity: 0.6404
MCC: 0.4887


**Will not attempt to use jackknife_validation because training time is 6hrs..**

But for any readers interested, below is the helper function to run jackknife

In [ ]:
def jackknife_validation(X_benchmark, y_benchmark, model):
  tp, tn, fp, fn = 0, 0, 0, 0
  for i in range(len(X_benchmark)):
    X_test, y_test = X_benchmark[i:i+1], y_benchmark[i:i+1]
    X_train = np.concatenate((X_benchmark[:i], X_benchmark[i+1:]), axis=0)
    y_train = np.concatenate((y_benchmark[0:i], y_benchmark[i+1:]), axis=0)

    cloned_model = clone(model)
    cloned_model.fit(X_train, y_train)
    pred, expected = cloned_model.predict(X_test)[0], y_test[0]

    if pred == expected and expected == 1:
      tp += 1
    elif pred == expected and expected == 0:
      tn += 1
    elif pred != expected and expected == 1:
      fn += 1
    elif pred != expected and expected == 0:
      fp += 1
    else:
      assert False, "Should not occur."

  print("Below are results on running jackknife validation on benchmark dataset")
  print(f"Accuracy: {PM.calculate_accuracy(tn+tp, tn+fp+fn+tp)}")
  print(f"Sensitivity: {PM.calculate_sensitivity(tp, fn)}")
  print(f"Specificity: {PM.calculate_specificity(tn, fp)}")
  print(f"MCC: {PM.calculate_MCC(tp, tn, fp, fn)}")

In [ ]:
# import warnings
# from sklearn.exceptions import ConvergenceWarning
# warnings.filterwarnings("ignore", category=ConvergenceWarning, module='sklearn.linear_model._logistic')

# jackknife_validation(X_pdb1075, y_pdb1075, StackDPPred(default_base_clf(), default_meta_clf()))

for stacking, is it alright if my base classifiers are trained on the full training set and then i obtained the results of the classifers on the training set to be combined with original features and passed as input features to the meta classifier?


so there's no need to hold out or split the training set to train the meta classifier on instances it has never seen before?

In the traditional stacking approach, the base classifiers are trained on the full training set, and their predictions are used as additional features to train the meta classifier. The meta classifier, however, is typically trained on a hold-out or validation set that is separate from the training set used for the base classifiers.

The purpose of training the meta classifier on a hold-out set is to ensure that it learns to generalize well to unseen data and avoids overfitting. By using a separate validation set, you can evaluate the performance of the meta classifier on data that it has not seen during training.



1. try out more LR iterations
2. Simply use SVM and nth else and compare against author's proposed model'
3. Do not use RCEM
4. Edit EDT and RPT


IGNORE JACKKNIFE VALIDATION!

Do two models in the end:
follow paper strictly (Edit EDT and RPT)
- try use SVM only

my own version (following paper strictly) (try more LR iterations, do not use RCEM)
- try use SVM only